# Sentiment head — completion run
New, unexecuted. Uses the original course dataset and grouped splits. This is a separate sequence-classification head, not a shared multitask encoder. Old metrics do not apply to the new preprocessing. Run in a fresh kernel. Outputs stay labelled MEASURED_SMOKE for the course sample.


In [ ]:

from pathlib import Path
import sys
ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT=ROOT.parent
assert (ROOT/'bayan_additions').is_dir(), 'Upload/clone the complete package and run from its root or notebooks directory.'
sys.path.insert(0,str(ROOT))
from bayan_additions.preprocessing import prepare, VERSION


In [ ]:
# تثبيت نسخ اليوم الثاني فقط عند الحاجة

import importlib.metadata
import importlib.util
import subprocess
import sys

REQUIRED = {
    "transformers": "5.15.1",
    "tokenizers": "0.22.2",
    "scikit-learn": "1.9.0",
}

needs_install = []

for distribution, expected in REQUIRED.items():
    try:
        current = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        current = None

    if current != expected:
        needs_install.append(f"{distribution}=={expected}")

if needs_install:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *needs_install
        ]
    )

if importlib.util.find_spec("torch") is None:
    raise RuntimeError(
        "PyTorch is required. Open this notebook in Google Colab."
    )

print("Python:", sys.version.split()[0])
print("Environment ready / البيئة جاهزة")

In [ ]:
import csv
import io
import json
import math
import os
import random
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, AutoTokenizer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

In [ ]:
DATA_URL = "https://raw.githubusercontent.com/almiyead-rgb/bayan-applied-nlp-course/main/data/sample/bayan_day2_classification.csv"

FALLBACK_ROWS = [
    {
        "example_id": "F-001",
        "group_id": "DG-A",
        "split": "train",
        "language": "ar",
        "text": "تعذر تسجيل الدخول إلى البوابة",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-002",
        "group_id": "DG-A",
        "split": "train",
        "language": "en",
        "text": "I cannot sign in to the portal",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-003",
        "group_id": "DG-B",
        "split": "train",
        "language": "ar",
        "text": "الخدمة الإلكترونية سريعة وواضحة",
        "topic": "digital_service",
        "sentiment": "positive",
    },
    {
        "example_id": "F-004",
        "group_id": "DG-B",
        "split": "train",
        "language": "en",
        "text": "The online service is clear and fast",
        "topic": "digital_service",
        "sentiment": "positive",
    },
    {
        "example_id": "F-005",
        "group_id": "PG-A",
        "split": "train",
        "language": "ar",
        "text": "أحتاج معرفة حالة طلب التصريح",
        "topic": "permit",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-006",
        "group_id": "PG-A",
        "split": "train",
        "language": "en",
        "text": "I need the status of my permit request",
        "topic": "permit",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-007",
        "group_id": "PG-B",
        "split": "train",
        "language": "ar",
        "text": "تمت الموافقة على التصريح اليوم",
        "topic": "permit",
        "sentiment": "positive",
    },
    {
        "example_id": "F-008",
        "group_id": "PG-B",
        "split": "train",
        "language": "en",
        "text": "The permit was approved today",
        "topic": "permit",
        "sentiment": "positive",
    },
    {
        "example_id": "F-009",
        "group_id": "HG-A",
        "split": "train",
        "language": "ar",
        "text": "تأخر موعد العيادة هذا الصباح",
        "topic": "health",
        "sentiment": "negative",
    },
    {
        "example_id": "F-010",
        "group_id": "HG-A",
        "split": "train",
        "language": "en",
        "text": "My clinic appointment was delayed",
        "topic": "health",
        "sentiment": "negative",
    },
    {
        "example_id": "F-011",
        "group_id": "HG-B",
        "split": "train",
        "language": "ar",
        "text": "كانت خدمة العيادة ممتازة",
        "topic": "health",
        "sentiment": "positive",
    },
    {
        "example_id": "F-012",
        "group_id": "HG-B",
        "split": "train",
        "language": "en",
        "text": "The clinic service was excellent",
        "topic": "health",
        "sentiment": "positive",
    },
    {
        "example_id": "F-013",
        "group_id": "TG-A",
        "split": "train",
        "language": "ar",
        "text": "الحافلة لم تصل في الوقت المحدد",
        "topic": "transport",
        "sentiment": "negative",
    },
    {
        "example_id": "F-014",
        "group_id": "TG-A",
        "split": "train",
        "language": "en",
        "text": "The bus did not arrive on time",
        "topic": "transport",
        "sentiment": "negative",
    },
    {
        "example_id": "F-015",
        "group_id": "TG-B",
        "split": "train",
        "language": "ar",
        "text": "كانت الرحلة مريحة ومنظمة",
        "topic": "transport",
        "sentiment": "positive",
    },
    {
        "example_id": "F-016",
        "group_id": "TG-B",
        "split": "train",
        "language": "en",
        "text": "The trip was comfortable and organised",
        "topic": "transport",
        "sentiment": "positive",
    },
    {
        "example_id": "F-017",
        "group_id": "DG-V",
        "split": "validation",
        "language": "ar",
        "text": "لم يصل رمز التحقق الرقمي",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-018",
        "group_id": "PG-V",
        "split": "validation",
        "language": "en",
        "text": "How can I renew the permit",
        "topic": "permit",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-019",
        "group_id": "HG-V",
        "split": "validation",
        "language": "ar",
        "text": "أحتاج إعادة جدولة الموعد الصحي",
        "topic": "health",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-020",
        "group_id": "TG-V",
        "split": "validation",
        "language": "en",
        "text": "The bus route has changed",
        "topic": "transport",
        "sentiment": "neutral",
    },
    {
        "example_id": "F-021",
        "group_id": "DG-T",
        "split": "test",
        "language": "en",
        "text": "The verification code did not arrive",
        "topic": "digital_service",
        "sentiment": "negative",
    },
    {
        "example_id": "F-022",
        "group_id": "PG-T",
        "split": "test",
        "language": "ar",
        "text": "تأخر إصدار التصريح المطلوب",
        "topic": "permit",
        "sentiment": "negative",
    },
    {
        "example_id": "F-023",
        "group_id": "HG-T",
        "split": "test",
        "language": "en",
        "text": "The clinic appointment was cancelled",
        "topic": "health",
        "sentiment": "negative",
    },
    {
        "example_id": "F-024",
        "group_id": "TG-T",
        "split": "test",
        "language": "ar",
        "text": "توقفت الحافلة قبل المحطة",
        "topic": "transport",
        "sentiment": "negative",
    },
]

try:
    with urllib.request.urlopen(DATA_URL, timeout=20) as response:
        text = response.read().decode("utf-8")
        rows = list(csv.DictReader(io.StringIO(text)))

    DATA_SOURCE = "github_course_file"

except Exception as exc:
    rows = FALLBACK_ROWS
    DATA_SOURCE = f"embedded_fallback:{type(exc).__name__}"

print("Data source:", DATA_SOURCE)
print("Rows:", len(rows))
print("Topics:", Counter(row["topic"] for row in rows))

assert len(rows) >= 24
assert {"ar", "en"} <= {row["language"] for row in rows}

In [ ]:
TASK = "sentiment"
for row in rows:
    row["raw_text"] = row["text"]
    row["text"] = prepare(row["text"])


In [ ]:
def validate_splits(rows):
    required = {"train", "validation", "test"}

    group_owner = {}
    labels_by_split = {
        name: set()
        for name in required
    }

    counts = Counter()

    for row in rows:
        split = row["split"]
        group = row["group_id"]
        label = row["sentiment"]

        if split not in required:
            raise ValueError(
                f"Unknown split: {split}"
            )

        previous = group_owner.setdefault(
            group,
            split
        )

        if previous != split:
            raise ValueError(
                f"Group leakage: {group}"
            )

        labels_by_split[split].add(label)
        counts[split] += 1

    all_labels = set().union(
        *labels_by_split.values()
    )

    for split in required:
        if split == "train" and labels_by_split[split] != all_labels:
            raise ValueError(
                f"Missing label in {split}"
            )

    return {
        "rows": dict(counts),
        "groups": len(group_owner),
        "labels": sorted(all_labels),
        "group_overlap": 0,
        "missing_labels_by_split": {k: sorted(all_labels-v) for k,v in labels_by_split.items()},
    }


split_report = validate_splits(rows)

print(
    json.dumps(
        split_report,
        ensure_ascii=False,
        indent=2
    )
)

assert split_report["group_overlap"] == 0

print("Split contract=PASS")

In [ ]:
train_rows = [row for row in rows if row["split"] == "train"]
validation_rows = [row for row in rows if row["split"] == "validation"]
test_rows = [row for row in rows if row["split"] == "test"]
LABELS = sorted({row["sentiment"] for row in rows})
label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

baseline = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1),
    LinearSVC(random_state=SEED),
)
baseline.fit(
    [row["text"] for row in train_rows],
    [row["sentiment"] for row in train_rows],
)
baseline_val_pred = baseline.predict([row["text"] for row in validation_rows])
baseline_val_f1 = f1_score(
    [row["sentiment"] for row in validation_rows],
    baseline_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
print("Baseline validation macro-F1 (MEASURED_SMOKE):", round(baseline_val_f1, 4))
assert 0.0 <= baseline_val_f1 <= 1.0
print("Baseline=PASS")

In [ ]:
MODEL_ID = "distilbert/distilbert-base-multilingual-cased"
MAX_LENGTH = 64
BATCH_SIZE = 4
NUM_EPOCHS = 2 if DEVICE.type == "cuda" else 12

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=len(LABELS),
        label2id=label2id,
        id2label=id2label,
    )
except Exception as exc:
    raise RuntimeError(
        "Checkpoint download failed. Reconnect the runtime and run this cell once. "
        "No API key is required."
    ) from exc

TRAINING_MODE = "full_finetune" if DEVICE.type == "cuda" else "partial_finetune_cpu"
if TRAINING_MODE == "partial_finetune_cpu":
    for parameter in model.base_model.parameters():
        parameter.requires_grad = False
    for parameter in model.base_model.transformer.layer[-1].parameters():
        parameter.requires_grad = True

model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Training mode:", TRAINING_MODE)
print("Epochs:", NUM_EPOCHS)
print(f"Trainable parameters: {trainable:,} / {total:,}")
assert trainable > 0

In [ ]:
def iter_batches(examples, batch_size, *, shuffle=False, seed=SEED):
    indexes = np.arange(len(examples))
    if shuffle:
        np.random.default_rng(seed).shuffle(indexes)
    for start in range(0, len(indexes), batch_size):
        yield [examples[index] for index in indexes[start:start + batch_size]]


def encode_batch(batch):
    encoded = tokenizer(
        [row["text"] for row in batch],
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded["labels"] = torch.tensor(
        [label2id[row["sentiment"]] for row in batch], dtype=torch.long
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def predict(examples):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in iter_batches(examples, BATCH_SIZE):
            encoded = encode_batch(batch)
            logits = model(**encoded).logits
            predictions.extend(logits.argmax(-1).cpu().tolist())
    return [id2label[index] for index in predictions]

print("Batch functions=PASS")

In [ ]:
learning_rate = 2e-5 if TRAINING_MODE == "full_finetune" else 1e-4
trainable_parameters = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_parameters, lr=learning_rate)
losses = []
train_steps = 0
epoch_history = []
best_validation_f1 = -1.0
best_epoch = 0
best_trainable_state = None

for epoch_index in range(NUM_EPOCHS):
    model.train()
    epoch_losses = []
    for batch in iter_batches(
        train_rows, BATCH_SIZE, shuffle=True, seed=SEED + epoch_index + 1
    ):
        optimizer.zero_grad(set_to_none=True)
        encoded = encode_batch(batch)
        output = model(**encoded)
        loss = output.loss
        if not torch.isfinite(loss):
            raise RuntimeError("Non-finite training loss")
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
        optimizer.step()
        loss_value = float(loss.detach().cpu())
        losses.append(loss_value)
        epoch_losses.append(loss_value)
        train_steps += 1

    epoch_predictions = predict(validation_rows)
    epoch_f1 = f1_score(
        [row["sentiment"] for row in validation_rows], epoch_predictions,
        labels=LABELS, average="macro", zero_division=0,
    )
    epoch_history.append({
        "epoch": epoch_index + 1,
        "mean_loss": float(np.mean(epoch_losses)),
        "validation_macro_f1": float(epoch_f1),
    })
    if epoch_f1 > best_validation_f1:
        best_validation_f1 = float(epoch_f1)
        best_epoch = epoch_index + 1
        if True:
            best_trainable_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.named_parameters()
                if parameter.requires_grad
            }
    print(
        f"epoch={epoch_index + 1:02d} mean_loss={np.mean(epoch_losses):.4f} "
        f"validation_macro_f1={epoch_f1:.4f}"
    )

if best_trainable_state is not None:
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name in best_trainable_state:
                parameter.copy_(best_trainable_state[name].to(parameter.device))
    selected_epoch = best_epoch
else:
    selected_epoch = NUM_EPOCHS

assert train_steps >= 1 and all(math.isfinite(value) for value in losses)
assert any(parameter.requires_grad for parameter in model.base_model.parameters())
print("Transformer optimizer steps=PASS", {"steps": train_steps, "selected_epoch": selected_epoch})

In [ ]:
transformer_val_pred = predict(validation_rows)
transformer_val_f1 = f1_score(
    [row["sentiment"] for row in validation_rows],
    transformer_val_pred,
    labels=LABELS,
    average="macro",
    zero_division=0,
)
validation_delta = transformer_val_f1 - baseline_val_f1
print("Selected epoch:", selected_epoch)
print("Transformer validation macro-F1 (MEASURED_SMOKE):", round(transformer_val_f1, 4))
print("Validation delta vs TF-IDF baseline:", round(validation_delta, 4))
print("Validation predictions:", list(zip(
    [row["sentiment"] for row in validation_rows], transformer_val_pred
)))
assert 0.0 <= transformer_val_f1 <= 1.0

In [ ]:
test_truth = [row["sentiment"] for row in test_rows]
baseline_test_pred = baseline.predict([row["text"] for row in test_rows]).tolist()
transformer_test_pred = predict(test_rows)

results = {
    "result_type": "MEASURED_SMOKE",
    "data_source": DATA_SOURCE,
    "model_id": MODEL_ID,
    "device": str(DEVICE),
    "training_mode": TRAINING_MODE,
    "seed": SEED,
    "epochs_run": NUM_EPOCHS,
    "selected_epoch": selected_epoch,
    "train_steps": train_steps,
    "mean_train_loss": float(np.mean(losses)),
    "baseline_validation_macro_f1": float(baseline_val_f1),
    "transformer_validation_macro_f1": float(transformer_val_f1),
    "validation_delta_vs_baseline": float(validation_delta),
    "baseline_beaten_on_validation": bool(validation_delta > 0),
    "baseline_test_macro_f1": float(f1_score(
        test_truth, baseline_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_macro_f1": float(f1_score(
        test_truth, transformer_test_pred, labels=LABELS,
        average="macro", zero_division=0,
    )),
    "transformer_test_accuracy": float(accuracy_score(test_truth, transformer_test_pred)),
    "limitations": [
        "synthetic tiny dataset",
        "small validation set used for epoch selection",
        "not an estimate of production quality",
    ],
}
print(json.dumps(results, ensure_ascii=False, indent=2))
Path(ROOT / "reports" / "new_sentiment_metrics.json").write_text(
    json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8"
)

In [ ]:

import hashlib
artifact_dir=ROOT/'artifacts'/TASK
artifact_dir.mkdir(parents=True,exist_ok=True)
model.save_pretrained(artifact_dir)
tokenizer.save_pretrained(artifact_dir)
exported=[]
model.eval()
for split_name, subset in [('validation',validation_rows),('test',test_rows)]:
    for row in subset:
        inputs=tokenizer(row['text'],truncation=True,max_length=MAX_LENGTH,return_tensors='pt')
        with torch.no_grad():
            probabilities=model(**{k:v.to(DEVICE) for k,v in inputs.items()}).logits.softmax(-1)[0].cpu()
        idx=int(probabilities.argmax())
        exported.append({**row,'label':row[TASK],'prediction_a':str(baseline.predict([row['text']])[0]),
            'prediction_b':id2label[idx],'confidence':float(probabilities[idx]),'task':TASK,
            'preprocessing_version':VERSION})
payload=json.dumps(rows,ensure_ascii=False,sort_keys=True).encode('utf-8')
manifest={'task':TASK,'data_kind':'MEASURED_SMOKE','preprocessing_version':VERSION,
    'data_sha256':hashlib.sha256(payload).hexdigest(),'data_source':DATA_SOURCE,
    'model_source':MODEL_ID,'max_length':MAX_LENGTH,'split_report':split_report,
    'selected_epoch':selected_epoch,'seed':SEED}
(artifact_dir/'training_manifest.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')
(ROOT/'reports'/f'{TASK}_predictions.json').write_text(json.dumps(exported,ensure_ascii=False,indent=2),encoding='utf-8')
(ROOT/'reports'/f'{TASK}_frozen_data.json').write_text(json.dumps(rows,ensure_ascii=False,indent=2),encoding='utf-8')
with (ROOT/'reports'/f'{TASK}_validation.csv').open('w',encoding='utf-8',newline='') as f:
    writer=csv.DictWriter(f,fieldnames=['example_id','split','language','text','label'])
    writer.writeheader()
    writer.writerows({k:r[k] for k in writer.fieldnames} for r in exported if r['split']=='validation')
print('New predictions, frozen data and trained checkpoint saved. Do not upload artifacts/ to GitHub.')


In [ ]:

# Audit the actual task tokenizer on each split before truncation is enabled per call.
token_audit={}
for split_name, subset in [('train',train_rows),('validation',validation_rows),('test',test_rows)]:
    lengths=[]; fertility=[]; unknown=[]
    for row in subset:
        encoded=tokenizer(row['text'],truncation=False,add_special_tokens=True)
        content=tokenizer(row['text'],truncation=False,add_special_tokens=False)['input_ids']
        lengths.append(len(encoded['input_ids']))
        fertility.append(len(content)/max(1,len(row['text'].split())))
        unknown.append(sum(t==tokenizer.unk_token_id for t in content)/max(1,len(content)))
    token_audit[split_name]={'n':len(subset),'mean_fertility':float(np.mean(fertility)),
        'mean_unk_fraction':float(np.mean(unknown)),'max_tokens':max(lengths),
        'p95_tokens':float(np.percentile(lengths,95)),
        'truncation_rate':sum(n>MAX_LENGTH for n in lengths)/len(lengths),'max_length':MAX_LENGTH}
(ROOT/'reports'/f'{TASK}_tokenization.json').write_text(json.dumps(token_audit,indent=2),encoding='utf-8')
print(token_audit)
